# CH01 -- Notebook thực hành
Chạy lần lượt từng ô từ trên xuống. Đặt notebook ở thư mục gốc dự án (cùng cấp với thư mục `data/` và `figures/`).

### Ô lệnh 1 -- Kiểm tra phiên bản Python và các thư viện chính

In [ ]:
import sys
import numpy, pandas, sklearn, matplotlib, xgboost, lightgbm

print("Python       :", sys.version.split()[0])
print("numpy        :", numpy.__version__)
print("pandas       :", pandas.__version__)
print("scikit-learn :", sklearn.__version__)
print("matplotlib   :", matplotlib.__version__)
print("xgboost      :", xgboost.__version__)
print("lightgbm     :", lightgbm.__version__)


### Ô lệnh 2 -- Nạp thư viện và thiết lập chung cho cả chương

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 80)       # mỗi dòng bảng in ra tối đa 80 ký tự
pd.set_option("display.max_columns", 8)  # hiện tối đa 8 cột, phần còn lại thay bằng "..."
pd.set_option("display.precision", 3)    # làm tròn 3 chữ số sau dấu phẩy

SEED = 42  # "hạt giống" ngẫu nhiên: cố định để mỗi lần chạy cho cùng một kết quả


### Ô lệnh 3 -- Đọc file Excel vào một bảng dữ liệu (DataFrame)

In [ ]:
df = pd.read_excel("data/credit_default.xls", header=1)

print("Số dòng, số cột:", df.shape)
print()
print(df.iloc[:5, :7])   # 5 dòng đầu, 7 cột đầu


### Ô lệnh 4 -- Kiểm tra kiểu dữ liệu, giá trị thiếu và dòng trùng lặp

In [ ]:
print("Kiểu dữ liệu của các cột:")
print(df.dtypes.value_counts())
print()
print("Tổng số ô bị thiếu (NaN):", df.isna().sum().sum())
print("Số dòng trùng lặp (bỏ qua cột ID):", df.drop(columns="ID").duplicated().sum())


### Ô lệnh 5 -- Bỏ cột ID và đặt lại tên cột cho dễ dùng

In [ ]:
df = df.drop(columns="ID")
df = df.rename(columns={"PAY_0": "PAY_1",
                        "default payment next month": "default"})

cols = df.columns.tolist()
for i in range(0, len(cols), 6):      # in 6 tên cột trên mỗi dòng cho gọn
    print(cols[i:i + 6])


### Ô lệnh 6 -- Biến mục tiêu: có bao nhiêu khách hàng vỡ nợ?

In [ ]:
counts = df["default"].value_counts()
ratio = df["default"].value_counts(normalize=True)
print(pd.DataFrame({"Số lượng": counts, "Tỷ lệ": ratio}))

fig, ax = plt.subplots(figsize=(4, 2.6))
ax.bar(["0: không vỡ nợ", "1: vỡ nợ"], counts.sort_index().values,
       color=["#2a78d6", "#eb6834"], width=0.5)
ax.set_ylabel("Số khách hàng")
plt.savefig("figures/ch01_s06_target.pdf", bbox_inches="tight")  # Jupyter: plt.show()


### Ô lệnh 7 -- Ba biến phân loại nhận những giá trị nào?

In [ ]:
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(col, "->", df[col].value_counts().sort_index().to_dict())


### Ô lệnh 8 -- Gộp các mã không có trong từ điển dữ liệu vào nhóm "khác"

In [ ]:
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})   # 4 = khác
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})                 # 3 = khác

for col in ["EDUCATION", "MARRIAGE"]:
    print(col, "->", df[col].value_counts().sort_index().to_dict())


### Ô lệnh 9 -- Trạng thái trả nợ tháng 9 (PAY_1) nhận những giá trị nào?

In [ ]:
print(df["PAY_1"].value_counts().sort_index().to_dict())


### Ô lệnh 10 -- Thống kê mô tả cho bốn biến số tiêu biểu

In [ ]:
num_cols = ["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1"]
desc = df[num_cols].describe().T            # .T: xoay bảng để mỗi biến là một dòng
print(desc[["mean", "std", "min", "50%", "max"]].round(0))


### Ô lệnh 11 -- Vẽ phân bố (histogram) của bốn biến số

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.4))
for ax, col in zip(axes, num_cols):
    ax.hist(df[col], bins=40, color="#2a78d6")
    ax.set_title(col)
    ax.locator_params(axis="x", nbins=3)     # chỉ hiện khoảng 3 vạch số cho đỡ rối
plt.savefig("figures/ch01_s11_hist.pdf", bbox_inches="tight")


### Ô lệnh 12 -- Tỷ lệ vỡ nợ theo trạng thái trả nợ tháng 9

In [ ]:
rate = df.groupby("PAY_1")["default"].agg(["count", "mean"])
rate.columns = ["Số khách hàng", "Tỷ lệ vỡ nợ"]
print(rate)

fig, ax = plt.subplots(figsize=(5.5, 2.6))
ax.bar(rate.index.astype(str), rate["Tỷ lệ vỡ nợ"], color="#2a78d6")
ax.axhline(df["default"].mean(), color="gray", ls="--")   # đường tỷ lệ chung 22%
ax.set_xlabel("PAY_1")
ax.set_ylabel("Tỷ lệ vỡ nợ")
plt.savefig("figures/ch01_s12_rate.pdf", bbox_inches="tight")


### Ô lệnh 13 -- Tách đặc trưng X và nhãn y, rồi chia train/test theo tỷ lệ 80/20

In [ ]:
from sklearn.model_selection import train_test_split

y = df["default"]
X = df.drop(columns="default").astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print("X_train:", X_train.shape, "   X_test:", X_test.shape)
print(f"Tỷ lệ vỡ nợ -- toàn bộ: {y.mean():.4f} | train: {y_train.mean():.4f}"
      f" | test: {y_test.mean():.4f}")


### Ô lệnh 14 -- Mô hình "ngây thơ": lúc nào cũng đoán lớp xuất hiện nhiều nhất

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

print("Các dự đoán:", pd.Series(pred_dummy).value_counts().to_dict())
print("Accuracy   :", round(accuracy_score(y_test, pred_dummy), 4))


### Ô lệnh 15 -- Hồi quy logistic, có chuẩn hóa dữ liệu trước khi học

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logreg.fit(X_train, y_train)
pred_lr = logreg.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred_lr), 4))
print("Số khách hàng bị dự đoán là vỡ nợ:", pred_lr.sum(), "/", len(pred_lr))
print("Số khách hàng thực sự vỡ nợ      :", y_test.sum())


### Ô lệnh 16 -- Ma trận nhầm lẫn của hồi quy logistic

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred_lr)
print(pd.DataFrame(cm, index=["Thực tế 0", "Thực tế 1"],
                   columns=["Dự đoán 0", "Dự đoán 1"]))

TN, FP, FN, TP = cm.ravel()       # "trải phẳng" ma trận 2x2 thành 4 con số
print(f"\nTN = {TN}, FP = {FP}, FN = {FN}, TP = {TP}")


### Ô lệnh 17 -- Tính Precision, Recall, F1 bằng tay rồi đối chiếu với scikit-learn

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"Tính tay    : Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}")
print(f"scikit-learn: Precision = {precision_score(y_test, pred_lr):.4f}, "
      f"Recall = {recall_score(y_test, pred_lr):.4f}, F1 = {f1_score(y_test, pred_lr):.4f}")


### Ô lệnh 18 -- Xác suất vỡ nợ mà mô hình gán cho 8 khách hàng đầu tiên của tập test

In [ ]:
proba_lr = logreg.predict_proba(X_test)[:, 1]   # cột 1 = xác suất thuộc lớp 1 (vỡ nợ)

view = pd.DataFrame({"Xác suất vỡ nợ": proba_lr[:8],
                     "Dự đoán (ngưỡng 0,5)": pred_lr[:8],
                     "Thực tế": y_test.values[:8]})
print(view)


### Ô lệnh 19 -- Cùng một mô hình, đổi ngưỡng quyết định thì kết quả thay đổi thế nào?

In [ ]:
rows = []
for t in [0.5, 0.4, 0.3, 0.2]:
    pred_t = (proba_lr >= t).astype(int)          # 1 nếu xác suất >= ngưỡng t
    rows.append({"Ngưỡng": t,
                 "Số bị gắn cờ": pred_t.sum(),
                 "Precision": precision_score(y_test, pred_t),
                 "Recall": recall_score(y_test, pred_t),
                 "F1": f1_score(y_test, pred_t)})
print(pd.DataFrame(rows).to_string(index=False))


### Ô lệnh 20 -- ROC-AUC và PR-AUC: đánh giá mô hình trên MỌI ngưỡng cùng lúc

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

print("ROC-AUC:", round(roc_auc_score(y_test, proba_lr), 4), "  (đoán ngẫu nhiên: 0,5)")
print("PR-AUC :", round(average_precision_score(y_test, proba_lr), 4),
      f"  (đoán ngẫu nhiên: {y_test.mean():.3f})")

fpr, tpr, _ = roc_curve(y_test, proba_lr)
prec, rec, _ = precision_recall_curve(y_test, proba_lr)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(fpr, tpr)
ax[0].plot([0, 1], [0, 1], "--", color="gray")
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR (= Recall)"); ax[0].set_title("Đường ROC")
ax[1].plot(rec, prec)
ax[1].axhline(y_test.mean(), ls="--", color="gray")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Đường Precision-Recall")
plt.savefig("figures/ch01_s20_curves.pdf", bbox_inches="tight")


### Ô lệnh 21 -- Đánh giá chéo 5-fold, viết vòng lặp "bằng tay" để thấy rõ từng bước

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = []
for k, (idx_tr, idx_va) in enumerate(skf.split(X_train, y_train), start=1):
    model = clone(logreg)                       # bản sao "mới tinh", chưa học gì
    model.fit(X_train.iloc[idx_tr], y_train.iloc[idx_tr])
    p = model.predict_proba(X_train.iloc[idx_va])[:, 1]
    s = average_precision_score(y_train.iloc[idx_va], p)
    scores.append(s)
    print(f"Fold {k}: học {len(idx_tr)} mẫu, kiểm tra {len(idx_va)} mẫu -> PR-AUC = {s:.4f}")

print(f"\nTrung bình ± độ lệch chuẩn: {np.mean(scores):.4f} ± {np.std(scores, ddof=1):.4f}")


### Ô lệnh 22 -- Viết gọn bằng cross_validate và tính nhiều chỉ số một lúc

In [ ]:
from sklearn.model_selection import cross_validate

scoring = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision",
           "F1": "f1", "Recall": "recall"}
res = cross_validate(logreg, X_train, y_train, cv=skf, scoring=scoring)

for name in scoring:
    s = res["test_" + name]                     # mảng 5 điểm số, mỗi fold một điểm
    print(f"{name:8s}: {s.mean():.4f} ± {s.std(ddof=1):.4f}")


### Ô lệnh 23 -- So sánh bốn mô hình bằng cùng một quy trình đánh giá chéo

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = {
    "LogReg": logreg,
    "RandomForest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                           n_jobs=2, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8,
                             eval_metric="logloss", n_jobs=2, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                               n_jobs=2, random_state=SEED, verbose=-1),
}
table = {}
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=skf, scoring=scoring)
    table[name] = {k: f"{r['test_' + k].mean():.3f} ± {r['test_' + k].std(ddof=1):.3f}"
                   for k in scoring}
print(pd.DataFrame(table).T)


### Ô lệnh 24 -- Đánh giá MỘT LẦN trên tập test (sau khi đã so sánh xong bằng CV)

In [ ]:
rows = {}
for name, m in models.items():
    m.fit(X_train, y_train)                     # học lại trên toàn bộ tập train
    p = m.predict_proba(X_test)[:, 1]
    rows[name] = {"ROC-AUC": roc_auc_score(y_test, p),
                  "PR-AUC": average_precision_score(y_test, p),
                  "F1": f1_score(y_test, (p >= 0.5).astype(int)),
                  "Recall": recall_score(y_test, (p >= 0.5).astype(int))}
print(pd.DataFrame(rows).T)


### Ô lệnh 25 -- Chênh lệch giữa hai mô hình có phải do may rủi? Kiểm định Wilcoxon

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import wilcoxon

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)  # 15 lần chia
auc = {name: cross_val_score(models[name], X_train, y_train, cv=rcv, scoring="roc_auc")
       for name in ["LogReg", "XGBoost", "LightGBM"]}
for name, s in auc.items():
    print(f"{name:8s}: ROC-AUC = {s.mean():.4f} ± {s.std(ddof=1):.4f}")
print()
for a, b in [("XGBoost", "LogReg"), ("XGBoost", "LightGBM")]:
    d = auc[a] - auc[b]
    p = wilcoxon(auc[a], auc[b]).pvalue
    print(f"{a} - {b}: chênh lệch TB = {d.mean():+.4f}, thắng {np.sum(d > 0)}/15 lần,"
          f" p = {p:.1e}")


### Ô lệnh 26 -- Bộ dữ liệu thứ hai: California Housing (có sẵn trong scikit-learn)

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)   # lần đầu sẽ tự tải về máy
Xh, yh = housing.data, housing.target

print("Kích thước:", Xh.shape)
print(Xh.head(3).round(2).T)        # .T: xoay bảng cho gọn
print()
print(yh.describe().round(3))


### Ô lệnh 27 -- Có bao nhiêu khu dân cư có giá bị "chặn trên"?

In [ ]:
n_cap = (yh >= 5.0).sum()
print(f"Số khu có giá >= 5,0: {n_cap} ({n_cap / len(yh):.1%})")
print("Giá trị lớn nhất    :", yh.max())

fig, ax = plt.subplots(figsize=(5, 2.6))
ax.hist(yh, bins=50, color="#2a78d6")
ax.set_xlabel("MedHouseVal (trăm nghìn USD)")
ax.set_ylabel("Số khu dân cư")
plt.savefig("figures/ch01_s27_hist.pdf", bbox_inches="tight")


### Ô lệnh 28 -- Tính MAE, RMSE và R² bằng tay trên ví dụ 4 căn nhà

In [ ]:
y_true = np.array([2.0, 3.0, 1.5, 4.0])     # giá thật
y_pred = np.array([2.5, 2.5, 1.5, 3.0])     # giá mô hình dự đoán
err = y_true - y_pred

print("Sai số:", err)
print("MAE  =", np.mean(np.abs(err)))
print("RMSE =", np.sqrt(np.mean(err ** 2)).round(4))
print("R²   =", (1 - np.sum(err ** 2) / np.sum((y_true - y_true.mean()) ** 2)).round(4))


### Ô lệnh 29 -- Chia dữ liệu và so sánh ba mô hình hồi quy bằng 5-fold CV

In [ ]:
from sklearn.model_selection import KFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, yh, test_size=0.2, random_state=SEED)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

reg_models = {
    "Dummy (trung bình)": DummyRegressor(strategy="mean"),
    "LinearReg": make_pipeline(StandardScaler(), LinearRegression()),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              n_jobs=2, random_state=SEED, verbose=-1),
}
reg_scoring = {"RMSE": "neg_root_mean_squared_error",
               "MAE": "neg_mean_absolute_error", "R2": "r2"}
tab = {}
for name, m in reg_models.items():
    r = cross_validate(m, Xh_train, yh_train, cv=kf, scoring=reg_scoring)
    tab[name] = {"RMSE": -r["test_RMSE"].mean(),   # đổi dấu: scikit-learn trả về số âm
                 "MAE": -r["test_MAE"].mean(),
                 "R2": r["test_R2"].mean()}
print(pd.DataFrame(tab).T.round(3))


### Ô lệnh 30 -- Đánh giá LightGBM trên tập test và xem vài dự đoán cụ thể

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

lgb_reg = reg_models["LightGBM"].fit(Xh_train, yh_train)
pred_h = lgb_reg.predict(Xh_test)
print(f"Test RMSE = {root_mean_squared_error(yh_test, pred_h):.3f},"
      f"  R2 = {r2_score(yh_test, pred_h):.3f}")
print()
print(pd.DataFrame({"Giá thật": yh_test.values[:6], "Dự đoán": pred_h[:6],
                    "Sai số": yh_test.values[:6] - pred_h[:6]}).round(3))
